# Getting Started with Alfred on Databricks (Free Edition)

**Alfred** is an open-source application designed to interact with data stored in Databricks.  

> ⚠️ **Important:** This notebook **must** be run in a Databricks environment. It will guide you through loading your first dataset in catalog within a schema.

---

### Step 0: Create a Free Databricks Account (if needed)

If you don’t have access yet, you can sign up for a free edition here:  
[Databricks Free Edition](https://www.databricks.com/learn/free-edition)

### Step 1: Access

Create and update your .env file with your sql warehouse credentials including your personal access token.

### Step 2: Run this notebook

### Step 3: Add metadata

> ⚠️ **Important:** After executing this notebook, add metadata to your data.


---

### Source Data

The sample dataset used in this notebook is available here:  
[GitHub: Czech Financial Data](https://github.com/kennethleungty/Text-to-SQL-with-KG-Neo4j-GraphRAG/blob/main/data/czech_financial.sqlite)

This notebook will help you explore the data, extract metadata, and prepare it for the construction of the knowledge graph.

In [ ]:
# Download SQLite DB
import requests, os

url = "https://github.com/kennethleungty/Text-to-SQL-with-KG-Neo4j-GraphRAG/raw/main/data/czech_financial.sqlite"
db_path = "/tmp/czech_financial.sqlite"

r = requests.get(url)
with open(db_path, "wb") as f:
    f.write(r.content)

db_path

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(db_path)

# Alle Tabellen anzeigen
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table';", conn
)

tables

In [ ]:
catalog = "alfred-app"
schema = "finance"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

print("If you use this notebook, catalog and schema must be set to alfred-app and finance respectively for the knowledge graph construction and for giving Alfred access to the data.")

In [ ]:
for t in tables["name"]:
    print(f"Loading {t}")

    # Escape reserved keywords by quoting the table name in SQLite
    pdf = pd.read_sql(f'SELECT * FROM "{t}"', conn)
    
    # Convert to Spark DataFrame
    sdf = spark.createDataFrame(pdf)
    
    # Save to Unity Catalog, escape table name with backticks for Spark SQL
    sdf.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.`{t}`")